# EDA — Franchise B (Quick Commerce / Dark Stores)

**Data format:** PSV (`|` delimiter, UTF-8 BOM) · Monthly + daily sales files · Hub master · Barcode-SKU mapping · Product master from manufacturer

**Key traits:** Uses internal warehouse barcodes (not SKU codes) · Barcode→SKU mapping required · Anonymous buyers (no customer IDs) · Net price = MRP − discount · Statuses: DELIVERED / RETURNED / CANCELLED · ~0.5% deliberate data-quality errors

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
%matplotlib inline

BASE = os.path.join('..', 'data_generator', 'storage', 'landing_zone')

## 1 · Load Data

In [ ]:
# Master data
product = pd.read_csv(os.path.join(BASE, 'master_company', 'product_master.csv'), encoding='utf-8-sig')
hubs = pd.read_csv(os.path.join(BASE, 'franchise_b', 'hubmaster_weekly_refresh.psv'), sep='|', encoding='utf-8-sig')
barcode_map = pd.read_csv(os.path.join(BASE, 'franchise_b', 'barcode_sku_mapping.psv'), sep='|', encoding='utf-8-sig')

# Concatenate all monthly + daily sales PSV files
sales_files = sorted(glob.glob(os.path.join(BASE, 'franchise_b', 'sales_*.psv')))
print(f'Sales files found: {len(sales_files)}')
for f in sales_files:
    print(f'  {os.path.basename(f)}')

sales = pd.concat(
    [pd.read_csv(f, sep='|', encoding='utf-8-sig') for f in sales_files],
    ignore_index=True
)
sales['delivery_timestamp'] = pd.to_datetime(sales['delivery_timestamp'])
print(f'\nTotal sales rows: {len(sales):,}')

## 2 · Schema & Basic Profiling

In [ ]:
print('=== Sales ===')
display(sales.info())
display(sales.describe())
display(sales.head())

In [ ]:
print('=== Hub Master ===')
display(hubs.info())
display(hubs.head())

print('\n=== Barcode-SKU Mapping ===')
display(barcode_map.info())
display(barcode_map.head())
print(f'\nUnique SKUs mapped: {barcode_map["sku_code"].nunique()}')
print(f'Barcodes per SKU: {barcode_map.groupby("sku_code").size().describe()}')
print(f'Active mappings: {barcode_map["is_active"].sum()} / {len(barcode_map)}')

In [ ]:
# Null counts
print('Null counts — Sales')
display(sales.isnull().sum())

print('\nNull counts — Barcode Mapping')
display(barcode_map.isnull().sum())

## 3 · Data Quality Checks (Quarantine Rules)

| Rule | Description |
|------|-------------|
| B1 | `mrp_price` must be > 0 |
| B2 | `discount_applied` cannot be negative or > `mrp_price` |
| B3 | `delivery_timestamp` cannot be in the future |
| B4 | `order_uuid` must be unique within the dataset |
| B5 | `item_barcode` must exist in mapping and be `is_active = 1` |

In [ ]:
now = pd.Timestamp.now()
active_barcodes = set(barcode_map.loc[barcode_map['is_active'] == 1, 'item_barcode'])

qa = pd.DataFrame({
    'B1_zero_mrp': sales['mrp_price'] <= 0,
    'B2_bad_discount': (sales['discount_applied'] < 0) | (sales['discount_applied'] > sales['mrp_price']),
    'B3_future_date': sales['delivery_timestamp'] > now,
    'B4_dup_uuid': sales.duplicated(subset='order_uuid', keep=False),
    'B5_unknown_barcode': ~sales['item_barcode'].isin(active_barcodes),
})
qa['any_error'] = qa.any(axis=1)

print('=== Quarantine Summary ===')
display(qa.sum())
print(f'\nError rate: {qa["any_error"].mean():.4%}')

In [ ]:
# Sample quarantined rows
display(sales[qa['any_error']].head(10))

In [ ]:
# Separate clean data
clean = sales[~qa['any_error']].copy()
print(f'Clean rows: {len(clean):,} / {len(sales):,}')

## 4 · Barcode → SKU → Product Enrichment

In [ ]:
# Map barcode to SKU
enriched = clean.merge(
    barcode_map[barcode_map['is_active'] == 1][['item_barcode', 'sku_code']].drop_duplicates('item_barcode'),
    on='item_barcode', how='left'
)

# Join product master
enriched = enriched.merge(
    product[['sku_code', 'brand', 'category', 'sub_category', 'size', 'color', 'fabric', 'mrp']],
    on='sku_code', how='left'
)

# Calculate net price
enriched['net_price'] = enriched['mrp_price'] - enriched['discount_applied']
enriched['revenue'] = enriched['units_sold'] * enriched['net_price']
enriched['month'] = enriched['delivery_timestamp'].dt.to_period('M')

print(f'Enriched rows: {len(enriched):,}')
print(f'SKU match rate: {enriched["sku_code"].notna().mean():.2%}')
display(enriched.head())

## 5 · Sales Analysis

In [ ]:
# Monthly revenue trend
monthly = enriched.groupby('month')['revenue'].sum()
ax = monthly.plot(kind='bar', figsize=(12, 4), title='Monthly Revenue — Franchise B')
ax.set_ylabel('Revenue (₹)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Revenue by category
cat_rev = enriched.groupby('category')['revenue'].sum().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cat_rev.plot(kind='bar', ax=axes[0], title='Revenue by Category')
cat_rev.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', title='Category Mix')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 hubs by revenue
hub_rev = (enriched.merge(hubs, on='hub_id', how='left')
           .groupby(['hub_id', 'hub_name'])['revenue'].sum()
           .sort_values(ascending=False).head(10))
display(hub_rev.reset_index())

In [ ]:
# Top sizes and colors by units sold
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
enriched.groupby('size')['units_sold'].sum().sort_values(ascending=False).head(10).plot(
    kind='barh', ax=axes[0], title='Top 10 Sizes by Units Sold')
enriched.groupby('color')['units_sold'].sum().sort_values(ascending=False).head(10).plot(
    kind='barh', ax=axes[1], title='Top 10 Colors by Units Sold')
plt.tight_layout()
plt.show()

## 6 · Order Status & Returns Analysis

In [ ]:
status_counts = enriched['status'].value_counts()
print('Order status distribution:')
display(status_counts)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
status_counts.plot(kind='bar', ax=axes[0], title='Order Count by Status')

# Revenue by status
enriched.groupby('status')['revenue'].sum().plot(kind='bar', ax=axes[1], title='Revenue by Status')
plt.tight_layout()
plt.show()

print(f'Return rate: {(enriched["status"] == "RETURNED").mean():.2%}')
print(f'Cancel rate: {(enriched["status"] == "CANCELLED").mean():.2%}')

## 7 · Discount Analysis

In [ ]:
delivered = enriched[enriched['status'] == 'DELIVERED'].copy()
delivered['discount_pct'] = delivered['discount_applied'] / delivered['mrp_price'] * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
delivered['discount_pct'].hist(bins=50, ax=axes[0])
axes[0].set_title('Discount % Distribution')
axes[0].set_xlabel('Discount %')

delivered.groupby('category')['discount_pct'].mean().sort_values(ascending=False).plot(
    kind='barh', ax=axes[1], title='Avg Discount % by Category')
plt.tight_layout()
plt.show()

## 8 · Hub Geo Distribution

In [ ]:
plt.figure(figsize=(6, 8))
plt.scatter(hubs['longitude'], hubs['latitude'], alpha=0.4, s=10)
plt.title('Hub Locations (lat/lon)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.show()

## 9 · Summary

| Metric | Value |
|--------|-------|
| Total rows loaded | see above |
| Quarantine error rate | ~0.5% |
| Barcode→SKU match rate | ~100% (active mappings) |
| Statuses | DELIVERED ~85%, RETURNED ~10%, CANCELLED ~5% |
| Key errors found | Zero MRP, duplicate UUIDs, negative discounts |